# TP4 - Grupo 5

In [ ]:
# importamos paquetes
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, LassoCV, RidgeCV
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, roc_auc_score, RocCurveDisplay, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
# ---------------------------------------------------------------
# Rutas del proyecto. El notebook vive en programs/ y lee de
# ../input/ y escribe en ../output/. Funciona tanto si se ejecuta
# desde programs/ como desde la raiz del TP.
# ---------------------------------------------------------------
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / "input").is_dir() else Path.cwd().parent
INPUT = BASE / "input"
OUTPUT = BASE / "output"
OUTPUT.mkdir(exist_ok=True)

print("Input :", INPUT)
print("Output:", OUTPUT)

In [ ]:
# importamos base de datos (se encuentra en la carpeta de github)
respondieron = pd.read_csv(INPUT / "respondieron.csv", low_memory = False)

## Métodos de regularización

In [ ]:
# volvemos a separar por 2004 y 2024, creamos los X train y X test
variables = ['CONSTANTE', 'CH04', 'CH06', 'CH07', 'CH16', 'PERSONAS_HOGAR', 'EDAD2', 'OBRA_SOCIAL', 'EDUCACION', 'EDADxEDUCACION', 'COLEGIO_PRIVADO']
nombres = ['CONSTANTE', 'SEXO', 'EDAD', 'ESTADO CIVIL', 'DONDE VIVÍA', 'PERSONAS HOGAR', 'EDAD^2', 'OBRA SOCIAL', 'AÑOS EDUCACIÓN', 'EDAD x EDUCACION', 'COLEGIO PRIVADO']

X_04 = respondieron.loc[respondieron['ANO4'] == 2004, variables].to_numpy()
y_04 = respondieron.loc[respondieron['ANO4'] == 2004, 'DESOCUPADOS'].to_numpy()

X_24 = respondieron.loc[respondieron['ANO4'] == 2024, variables].to_numpy()
y_24 = respondieron.loc[respondieron['ANO4'] == 2024, 'DESOCUPADOS'].to_numpy()

X_train_04, X_test_04, y_train_04, y_test_04 = train_test_split(X_04, y_04, test_size=0.3, random_state=444)
X_train_24, X_test_24, y_train_24, y_test_24 = train_test_split(X_24, y_24, test_size=0.3, random_state=444)

In [ ]:
# estandarizamos las X
sc_04 = StandardScaler()
X_train_04 = sc_04.fit_transform(X_train_04)
X_test_04 = sc_04.transform(X_test_04)

sc_24 = StandardScaler()
X_train_24 = sc_24.fit_transform(X_train_24)
X_test_24 = sc_24.transform(X_test_24)

### Ejercicio 3: penalizaciones LASSO y Ridge para regresión logística

#### 2004

In [ ]:
# modelo sin penalidad
log_reg_04 = LogisticRegression(penalty=None, max_iter=3000).fit(X_train_04, y_train_04)
y_predRL_04 = log_reg_04.predict(X_test_04)
y_probRL_04 = log_reg_04.predict_proba(X_test_04)[:,1]

matriz_confusionRL_04 = confusion_matrix(y_test_04, y_predRL_04)
accuracyRL_04 = accuracy_score(y_test_04, y_predRL_04)
aucRL_04 = roc_auc_score(y_test_04, y_probRL_04)
mseRL_04 = mean_squared_error(y_test_04, y_predRL_04)
fprRL_04, tprRL_04, thresholdsRL_04 = roc_curve(y_test_04, y_probRL_04, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionRL_04}')
print(f'El accuracy score es: {accuracyRL_04}.')
print(f'El área bajo la curva ROC es: {aucRL_04}.')
print(f'El error cuadrático medio de la muestra de testeo es: {mseRL_04}.')

In [ ]:
# graficamos su curva ROC
displayRL_04 = RocCurveDisplay(fpr=fprRL_04, tpr=tprRL_04, roc_auc=aucRL_04, estimator_name='RL')
displayRL_04.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "reg-log-2004.png", dpi=300, bbox_inches='tight')
plt.show()

#### 2024

In [ ]:
# modelo sin penalidad
log_reg_24 = LogisticRegression(penalty=None, max_iter=3000).fit(X_train_24, y_train_24)
y_predRL_24 = log_reg_24.predict(X_test_24)
y_probRL_24 = log_reg_24.predict_proba(X_test_24)[:,1]

matriz_confusionRL_24 = confusion_matrix(y_test_24, y_predRL_24)
accuracyRL_24 = accuracy_score(y_test_24, y_predRL_24)
aucRL_24 = roc_auc_score(y_test_24, y_probRL_24)
mseRL_24 = mean_squared_error(y_test_24, y_predRL_24)
fprRL_24, tprRL_24, thresholdsRL_24 = roc_curve(y_test_24, y_probRL_24, drop_intermediate=False)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionRL_24}')
print(f'El accuracy score es: {accuracyRL_24}.')
print(f'El área bajo la curva ROC es: {aucRL_24}.')
print(f'El error cuadrático medio de la muestra de testeo es: {mseRL_24}.')

In [ ]:
# graficamos su curva ROC
displayRL_24 = RocCurveDisplay(fpr=fprRL_24, tpr=tprRL_24, roc_auc=aucRL_24, estimator_name='RL')
displayRL_24.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "reg-log-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### LASSO

##### 2004

In [ ]:
# modelo
L_04 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=5000).fit(X_train_04, y_train_04)
y_predL_04 = L_04.predict(X_test_04)
y_probL_04 = L_04.predict_proba(X_test_04)[:,1]

matriz_confusionL_04 = confusion_matrix(y_test_04, y_predL_04)
accuracyL_04 = accuracy_score(y_test_04, y_predL_04)
aucL_04 = roc_auc_score(y_test_04, y_probL_04)
fprL_04, tprL_04, thresholdsL_04 = roc_curve(y_test_04, y_probL_04, drop_intermediate=False)
mseL_04 = mean_squared_error(y_test_04, y_predL_04)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionL_04}')
print(f'El accuracy score es: {accuracyL_04}.')
print(f'El área bajo la curva ROC es: {aucL_04}.')
print(f'El mean squared error de testeo es: {mseL_04}')

In [ ]:
# graficamos su curva ROC
displayL_04 = RocCurveDisplay(fpr=fprL_04, tpr=tprL_04, roc_auc=aucL_04, estimator_name='RL - LASSO')
displayL_04.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "lasso-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelo
L_24 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=5000).fit(X_train_24, y_train_24)
y_predL_24 = L_24.predict(X_test_24)
y_probL_24 = L_24.predict_proba(X_test_24)[:,1]

matriz_confusionL_24 = confusion_matrix(y_test_24, y_predL_24)
accuracyL_24 = accuracy_score(y_test_24, y_predL_24)
aucL_24 = roc_auc_score(y_test_24, y_probL_24)
fprL_24, tprL_24, thresholdsL_24 = roc_curve(y_test_24, y_probL_24, drop_intermediate=False)
mseL_24 = mean_squared_error(y_test_24, y_predL_24)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionL_24}')
print(f'El accuracy score es: {accuracyL_24}.')
print(f'El área bajo la curva ROC es: {aucL_24}.')
print(f'El mean squared error de testeo es: {mseL_24}.')

In [ ]:
# graficamos su curva ROC
displayL_24 = RocCurveDisplay(fpr=fprL_24, tpr=tprL_24, roc_auc=aucL_24, estimator_name='RL - LASSO')
displayL_24.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "lasso-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### Ridge

##### 2004

In [ ]:
# modelo
R_04 = LogisticRegression(penalty='l2', solver='liblinear', max_iter=5000).fit(X_train_04, y_train_04)
y_predR_04 = R_04.predict(X_test_04)
y_probR_04 = R_04.predict_proba(X_test_04)[:,1]

matriz_confusionR_04 = confusion_matrix(y_test_04, y_predR_04)
accuracyR_04 = accuracy_score(y_test_04, y_predR_04)
aucR_04 = roc_auc_score(y_test_04, y_probR_04)
fprR_04, tprR_04, thresholdsR_04 = roc_curve(y_test_04, y_probR_04, drop_intermediate=False)
mseR_04 = mean_squared_error(y_test_04, y_predR_04)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionR_04}')
print(f'El accuracy score es: {accuracyR_04}.')
print(f'El área bajo la curva ROC es: {aucR_04}.')
print(f'El mean squared error de testeo es: {mseR_04}.')

In [ ]:
# graficamos su curva ROC
displayR_04 = RocCurveDisplay(fpr=fprR_04, tpr=tprR_04, roc_auc=aucR_04, estimator_name='RL - Ridge')
displayR_04.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "ridge-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# modelo
R_24 = LogisticRegression(penalty='l2', solver='liblinear', max_iter=5000).fit(X_train_24, y_train_24)
y_predR_24 = R_24.predict(X_test_24)
y_probR_24 = R_24.predict_proba(X_test_24)[:,1]

matriz_confusionR_24 = confusion_matrix(y_test_24, y_predR_24)
accuracyR_24 = accuracy_score(y_test_24, y_predR_24)
aucR_24 = roc_auc_score(y_test_24, y_probR_24)
fprR_24, tprR_24, thresholdsR_24 = roc_curve(y_test_24, y_probR_24, drop_intermediate=False)
mseR_24 = mean_squared_error(y_test_24, y_predR_24)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusionR_24}')
print(f'El accuracy score es: {accuracyR_24}.')
print(f'El área bajo la curva ROC es: {aucR_24}.')
print(f'El mean squared error de testeo es: {mseR_24}.')  

In [ ]:
# graficamos su curva ROC
displayR_24 = RocCurveDisplay(fpr=fprR_24, tpr=tprR_24, roc_auc=aucR_24, estimator_name='RL - Ridge')
displayR_24.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "ridge-2024.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 4: validación cruzada de 10 folds para elegir el λ óptimo

In [ ]:
# creamos lista de lambdas
lambdas = 10.0**np.arange(-5, 6)

#### LASSO

##### 2004

In [ ]:
# hacemos cross validation y obtenemos lambda óptimo
cvL_04 = KFold(n_splits=10, shuffle=True, random_state=444)
mseL_04 = []
coeficientesL_04 = []

for a in lambdas:
    fold_mse = []
    fold_ceros = []

    Lasso_04 = Lasso(alpha=a, fit_intercept=False, max_iter=100000)

    for i, (train_index, valid_index) in enumerate(cvL_04.split(X_train_04)):
        x_train, x_valid = X_train_04[train_index], X_train_04[valid_index]
        y_train, y_valid = y_train_04[train_index], y_train_04[valid_index]

        LassoCV_04 = Lasso_04.fit(x_train, y_train)
        y_pred = LassoCV_04.predict(x_valid)
        
        mse = mean_squared_error(y_valid, y_pred)
        fold_mse.append(mse)

        prop_ceros = np.mean(LassoCV_04.coef_ == 0)
        fold_ceros.append(prop_ceros)

    mseL_04.append(np.mean(fold_mse))
    coeficientesL_04.append(np.mean(fold_ceros))

lambdaL_04 = lambdas[np.argmin(mseL_04)]
print(f'En 2004, el lambda óptimo en Lasso es: 10^({np.log10(lambdaL_04)}).')

In [ ]:
# graficamos promedio MSE test
plt.figure(figsize=(8, 6))
plt.plot(lambdas, mseL_04, label='MSE test promedio', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("MSE test")
plt.title("Promedio de los MSE test con CV - Lasso (2004)")
plt.grid(True)
plt.axis("tight")
plt.savefig(OUTPUT / "mse-lasso-2004.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# graficamos proporción variables descartadas
plt.figure(figsize=(8, 6))
plt.plot(lambdas, coeficientesL_04, label='variables descartadas', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("Proporción")
plt.title("Proporción de variables descartadas - Lasso (2004)")
plt.grid(True)
plt.ylim(0, 1.05)
plt.axis("tight")
plt.savefig(OUTPUT / "descartes-lasso-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# hacemos cross validation y obtenemos lambda óptimo
cvL_24 = KFold(n_splits=10, shuffle=True, random_state=444)
mseL_24 = []
coeficientesL_24 = []

for a in lambdas:
    fold_mse = []
    fold_ceros = []

    Lasso_24 = Lasso(alpha=a, fit_intercept=False, max_iter=100000)

    for i, (train_index, valid_index) in enumerate(cvL_24.split(X_train_24)):
        x_train, x_valid = X_train_24[train_index], X_train_24[valid_index]
        y_train, y_valid = y_train_24[train_index], y_train_24[valid_index]

        LassoCV_24 = Lasso_24.fit(x_train, y_train)
        y_pred = LassoCV_24.predict(x_valid)
        
        mse = mean_squared_error(y_valid, y_pred)
        fold_mse.append(mse)

        prop_ceros = np.mean(LassoCV_24.coef_ == 0)
        fold_ceros.append(prop_ceros)

    mseL_24.append(np.mean(fold_mse))
    coeficientesL_24.append(np.mean(fold_ceros))

lambdaL_24 = lambdas[np.argmin(mseL_24)]
print(f'En 2024, el lambda óptimo en Lasso es: 10^({np.log10(lambdaL_24)}).')

In [ ]:
# graficamos promedio MSE test
plt.figure(figsize=(8, 6))
plt.plot(lambdas, mseL_24, label='MSE test promedio', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("MSE test")
plt.title("Promedio de los MSE test con CV - Lasso (2024)")
plt.grid(True)
plt.axis("tight")
plt.savefig(OUTPUT / "mse-lasso-2024.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# graficamos proporción variables descartadas
plt.figure(figsize=(8, 6))
plt.plot(lambdas, coeficientesL_24, label='variables descartadas', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("Proporción")
plt.title("Proporción de variables descartadas - Lasso (2024)")
plt.grid(True)
plt.ylim(0, 1.05)
plt.axis("tight")
plt.savefig(OUTPUT / "descartes-lasso-2024.png", dpi=300, bbox_inches='tight')
plt.show()

#### Ridge

##### 2004

In [ ]:
# hacemos cross validation y obtenemos lambda óptimo
cvR_04 = KFold(n_splits=10, shuffle=True, random_state=444)
mseR_04 = []

for a in lambdas:
    fold_mse = []
    fold_ceros = []

    Ridge_04 = Ridge(alpha=a, fit_intercept=False, max_iter=100000)

    for i, (train_index, valid_index) in enumerate(cvR_04.split(X_train_04)):
        x_train, x_valid = X_train_04[train_index], X_train_04[valid_index]
        y_train, y_valid = y_train_04[train_index], y_train_04[valid_index]

        RidgeCV_04 = Ridge_04.fit(x_train, y_train)
        y_pred = RidgeCV_04.predict(x_valid)
        
        mse = mean_squared_error(y_valid, y_pred)
        fold_mse.append(mse)

    mseR_04.append(np.mean(fold_mse))

lambdaR_04 = lambdas[np.argmin(mseR_04)]
print(f'En 2004, el lambda óptimo en Ridge es: 10^({np.log10(lambdaR_04)}).')

In [ ]:
# graficamos promedio MSE test
plt.figure(figsize=(8, 6))
plt.plot(lambdas, mseR_04, label='MSE test promedio', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("MSE test")
plt.title("Promedio de los MSE test con CV - Ridge (2004)")
plt.grid(True)
plt.axis("tight")
plt.savefig(OUTPUT / "mse-ridge-2004.png", dpi=300, bbox_inches='tight')
plt.show()

##### 2024

In [ ]:
# hacemos cross validation y obtenemos lambda óptimo
cvR_24 = KFold(n_splits=10, shuffle=True, random_state=444)
mseR_24 = []

for a in lambdas:
    fold_mse = []
    fold_ceros = []

    Ridge_24 = Ridge(alpha=a, fit_intercept=False, max_iter=100000)

    for i, (train_index, valid_index) in enumerate(cvR_24.split(X_train_24)):
        x_train, x_valid = X_train_24[train_index], X_train_24[valid_index]
        y_train, y_valid = y_train_24[train_index], y_train_24[valid_index]

        RidgeCV_24 = Ridge_24.fit(x_train, y_train)
        y_pred = RidgeCV_24.predict(x_valid)
        
        mse = mean_squared_error(y_valid, y_pred)
        fold_mse.append(mse)
        
    mseR_24.append(np.mean(fold_mse))

lambdaR_24 = lambdas[np.argmin(mseR_24)]
print(f'En 2024, el lambda óptimo en Ridge es: 10^({np.log10(lambdaR_24)}).')

In [ ]:
# graficamos promedio MSE test
plt.figure(figsize=(8, 6))
plt.plot(lambdas, mseR_24, label='MSE test promedio', marker='o')
plt.xscale("log")
plt.xlabel("Lambdas")
plt.ylabel("MSE test")
plt.title("Promedio de los MSE test con CV - Ridge (2024)")
plt.grid(True)
plt.axis("tight")
plt.savefig(OUTPUT / "mse-ridge-2024.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicios 5 y 6: especificación con el λ óptimo

#### LASSO

##### 2004

In [ ]:
# estimamos el modelo
LassoO_04 = Lasso(alpha=lambdaL_04)
LassoO_04 = LassoO_04.fit(X_train_04, y_train_04)
y_predLO_04 = LassoO_04.predict(X_test_04)

mseLO_04 = mean_squared_error(y_test_04, y_predLO_04)
coeficientesLO_04 = LassoO_04.coef_

In [ ]:
# imprimimos resultados
print(f'En 2004, los coeficientes con Lasso fueron: {coeficientesLO_04}.')
print(f'El mean squared error de testeo es: {mseLO_04}.')

##### 2024

In [ ]:
# estimamos el modelo
LassoO_24 = Lasso(alpha=lambdaL_24)
LassoO_24 = LassoO_24.fit(X_train_24, y_train_24)
y_predLO_24 = LassoO_24.predict(X_test_24)

mseLO_24 = mean_squared_error(y_test_24, y_predLO_24)
coeficientesLO_24 = LassoO_24.coef_

In [ ]:
# imprimimos resultados
print(f'En 2024, los coeficientes con Lasso fueron: {coeficientesLO_24}.')
print(f'El mean squared error de testeo es: {mseLO_24}.')

#### Ridge

##### 2004

In [ ]:
# estimamos el modelo
RidgeO_04 = Ridge(alpha=lambdaR_04)
RidgeO_04 = RidgeO_04.fit(X_train_04, y_train_04)
y_predRO_04 = RidgeO_04.predict(X_test_04)

mseRO_04 = mean_squared_error(y_test_04, y_predRO_04)

In [ ]:
# imprimimos resultados
print(f'El mean squared error de testeo es: {mseRO_04}.')

##### 2024

In [ ]:
# estimamos el modelo
RidgeO_24 = Ridge(alpha=lambdaR_24)
RidgeO_24 = RidgeO_04.fit(X_train_24, y_train_24)
y_predRO_24 = RidgeO_04.predict(X_test_24)

mseRO_24 = mean_squared_error(y_test_24, y_predRO_24)

In [ ]:
# imprimimos resultados
print(f'El mean squared error de testeo es: {mseRO_24}.')

## Árboles

### Ejercicio 7: estimación árbol de decisión

In [ ]:
# volvemos a separar las bases, para usar los datos no escalados
X_train_04, X_test_04, y_train_04, y_test_04 = train_test_split(X_04, y_04, test_size=0.3, random_state=444)
X_train_24, X_test_24, y_train_24, y_test_24 = train_test_split(X_24, y_24, test_size=0.3, random_state=444)

#### 2004

In [ ]:
# elegimos max_depth óptima con CV
cv_04 = KFold(n_splits=10, shuffle=True, random_state=444)
max_attributes_04 = X_train_04.shape[1]
depth_range_04 = range(1, max_attributes_04 + 1)
accuracies_04 = []

for depth in depth_range_04:
    fold_accuracy = []
    tree_model = DecisionTreeClassifier(max_depth=depth, random_state=444)
        
    for train_idx, val_idx in cv_04.split(X_train_04):
        x_tr, x_val = X_train_04[train_idx], X_train_04[val_idx]
        y_tr, y_val = y_train_04[train_idx], y_train_04[val_idx]

        model = tree_model.fit(x_tr, y_tr)
        acc = model.score(x_val, y_val)
        fold_accuracy.append(acc)

    avg = np.mean(fold_accuracy)
    accuracies_04.append(avg)

df_04 = pd.DataFrame({"Max Depth": depth_range_04, "Average Accuracy": accuracies_04})
print(df_04.to_string(index=False))

max_depth_04 = np.argmax(accuracies_04) + 1
print(f'La max_depth óptima es: {max_depth_04}.')

In [ ]:
# hacemos el decision tree
decision_tree_04 = DecisionTreeClassifier(max_depth = max_depth_04, random_state=444).fit(X_train_04, y_train_04)

fig_04 = plt.figure(figsize=(20,15))
graph_24 = plot_tree(decision_tree_04,
                  max_depth = max_depth_04,
                  impurity = True,
                  feature_names = nombres,
                  class_names = ['No Desocupado', 'Desocupado'],
                  rounded = True, 
                  filled = True)

plt.savefig(OUTPUT / "arbol-2004.png", dpi=300, bbox_inches='tight')

#### 2024

In [ ]:
# elegimos max_depth óptima con CV
cv_24 = KFold(n_splits=10, shuffle=True, random_state=444)
max_attributes_24 = X_train_24.shape[1]
depth_range_24 = range(1, max_attributes_24 + 1)
accuracies_24 = []

for depth in depth_range_24:
    fold_accuracy = []
    tree_model = DecisionTreeClassifier(max_depth=depth, random_state=444)
        
    for train_idx, val_idx in cv_24.split(X_train_24):
        x_tr, x_val = X_train_24[train_idx], X_train_24[val_idx]
        y_tr, y_val = y_train_24[train_idx], y_train_24[val_idx]

        model = tree_model.fit(x_tr, y_tr)
        acc = model.score(x_val, y_val)
        fold_accuracy.append(acc)

    avg = np.mean(fold_accuracy)
    accuracies_24.append(avg)

df_24 = pd.DataFrame({"Max Depth": depth_range_24, "Average Accuracy": accuracies_24})
print(df_24.to_string(index=False))

max_depth_24 = np.argmax(accuracies_24) + 1
print(f'La max_depth óptima es: {max_depth_24}.')

In [ ]:
# hacemos el decision tree
decision_tree_24 = DecisionTreeClassifier(max_depth = max_depth_24, random_state=444).fit(X_train_24, y_train_24)

fig_24 = plt.figure(figsize=(20,15))
graph_24 = plot_tree(decision_tree_24,
                  max_depth = max_depth_24,
                  impurity = True,
                  feature_names = nombres,
                  class_names = ['No Desocupado', 'Desocupado'],
                  rounded = True, 
                  filled = True)

plt.savefig(OUTPUT / "arbol-2024.png", dpi=300, bbox_inches='tight')

### Ejercicio 8: gráfico importancia de los predictores

#### 2004

In [ ]:
# modelamos de nuevo para encontrar la importancias de los predictores
tree_04 = DecisionTreeClassifier(random_state=444).fit(X_train_04, y_train_04)
importancia_04 = tree_04.feature_importances_
df_importancias_04 = pd.DataFrame({"Variable": nombres, "Importancia": importancia_04}).sort_values(by="Importancia", ascending=False)

In [ ]:
# graficamos
plt.figure(figsize=(10, 6))
plt.barh(df_importancias_04["Variable"], df_importancias_04["Importancia"], color="steelblue")
plt.xlabel("Importancia")
plt.title("Importancia de los predictores - Árbol 2004")
plt.gca().invert_yaxis()
plt.grid(True, axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(OUTPUT / "importancia-2004.png", dpi=300, bbox_inches='tight')
plt.show()

#### 2024

In [ ]:
# encontramos importancias de predictores
tree_24 = DecisionTreeClassifier(random_state=444).fit(X_train_24, y_train_24)
importancia_24 = tree_24.feature_importances_
df_importancias_24 = pd.DataFrame({"Variable": nombres, "Importancia": importancia_24}).sort_values(by="Importancia", ascending=False)

In [ ]:
# graficamos
plt.figure(figsize=(10, 6))
plt.barh(df_importancias_24["Variable"], df_importancias_24["Importancia"], color="steelblue")
plt.xlabel("Importancia")
plt.title("Importancia de los predictores - Árbol 2024")
plt.gca().invert_yaxis()
plt.grid(True, axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(OUTPUT / "importancia-2024.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 9: medidas de precisión

#### 2004

In [ ]:
# calculamos medidas
y_pred_04 = decision_tree_04.predict(X_test_04)
y_prob_04 = decision_tree_04.predict_proba(X_test_04)[:, 1]

matriz_confusion_04 = confusion_matrix(y_test_04, y_pred_04)
accuracy_04 = accuracy_score(y_test_04, y_pred_04)
auc_04 = roc_auc_score(y_test_04, y_prob_04)
fpr_04, tpr_04, thresholds_04 = roc_curve(y_test_04, y_prob_04, drop_intermediate=False)
mse_04 = mean_squared_error(y_test_04, y_pred_04)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusion_04}')
print(f'El accuracy score es: {accuracy_04}.')
print(f'El área bajo la curva ROC es: {auc_04}.')
print(f'El mean squared error de testeo es: {mse_04}.')

In [ ]:
# graficamos su curva ROC
display_04 = RocCurveDisplay(fpr=fpr_04, tpr=tpr_04, roc_auc=auc_04, estimator_name='Decision Tree')
display_04.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "curva-arbol-2004.png", dpi=300, bbox_inches='tight')
plt.show()

#### 2024

In [ ]:
# calculamos medidas
y_pred_24 = decision_tree_24.predict(X_test_24)
y_prob_24 = decision_tree_24.predict_proba(X_test_24)[:, 1]

matriz_confusion_24 = confusion_matrix(y_test_24, y_pred_24)
accuracy_24 = accuracy_score(y_test_24, y_pred_24)
auc_24 = roc_auc_score(y_test_24, y_prob_24)
fpr_24, tpr_24, thresholds_24 = roc_curve(y_test_24, y_prob_24, drop_intermediate=False)
mse_24 = mean_squared_error(y_test_24, y_pred_24)

In [ ]:
# imprimimos resultados
print(f'La matriz de confusión es: \n {matriz_confusion_24}')
print(f'El accuracy score es: {accuracy_24}.')
print(f'El área bajo la curva ROC es: {auc_24}.')
print(f'El mean squared error de testeo es: {mse_24}.')

In [ ]:
# graficamos su curva ROC
display_24 = RocCurveDisplay(fpr=fpr_24, tpr=tpr_24, roc_auc=auc_24, estimator_name='Decision Tree')
display_24.plot()  
plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=.5)
plt.savefig(OUTPUT / "curva-arbol-2024.png", dpi=300, bbox_inches='tight')
plt.show()

FIN :)